# 🥇 Gold Layer: Business Analytics & KPIs

**Objective:** Aggregate the cleansed Silver data into highly optimized, business-level Key Performance Indicators (KPIs) ready for Business Intelligence consumption.

### Architecture Details
* **State Filtering:** All transformations explicitly filter for `is_active == True` to ensure aggregations are only performed on the most current data state.
* **BI Optimization:** These tables are constructed as dimensional snapshots and fact aggregations. They are designed to be queried directly by **Power BI** via the Databricks Serverless SQL Endpoint using DirectQuery or Import mode.
* **Core Metrics Generated:**
  1. Inventory Snapshots (Total vs. Damaged)
  2. Supplier Performance (Delivery Delays)
  3. Low Stock Alerts (Reorder Thresholds)
  4. Product Movement & Sales Trends

In [0]:
from pyspark.sql.functions import sum, count, col, when

base_path = "abfss://iaspcontainer@iaspstorage.dfs.core.windows.net"

print("Initializing Gold Layer Analytics...")

# ---------------------------------------------------------
# 1. Create Inventory Summary Table
# ---------------------------------------------------------
try:
    print("Aggregating: gold_inventory_summary...")
    
    # Read the clean Silver inventory table
    silver_inventory = spark.read.format("delta").table("silver_inventory")
    
    # Aggregate total stock and damaged stock per product
    inventory_summary = (silver_inventory
                         .groupBy("product_id")
                         .agg(
                             sum("stock_quantity").alias("total_stock_available"),
                             sum("damaged_quantity").alias("total_damaged_stock")
                         ))
    
    # Write to Gold layer (Overwriting with fresh calculations)
    (inventory_summary.write
     .format("delta")
     .mode("overwrite") 
     .option("path", f"{base_path}/gold/inventory_summary/")
     .saveAsTable("gold_inventory_summary"))
     
except Exception as e:
    print(f"Error processing inventory_summary: {e}")

# ---------------------------------------------------------
# 2. Create Supplier Performance Table
# ---------------------------------------------------------
try:
    print("Aggregating: gold_supplier_performance...")
    
    # Read the clean Silver shipments table
    silver_shipments = spark.read.format("delta").table("silver_shipments")
    
    # Calculate total shipments and count how many were delayed per supplier
    supplier_performance = (silver_shipments
                            .groupBy("supplier_id")
                            .agg(
                                count("shipment_id").alias("total_shipments"),
                                sum(when(col("delay_days") > 0, 1).otherwise(0)).alias("delayed_shipments")
                            ))
    
    # Write to Gold layer
    (supplier_performance.write
     .format("delta")
     .mode("overwrite")
     .option("path", f"{base_path}/gold/supplier_performance/")
     .saveAsTable("gold_supplier_performance"))
     
except Exception as e:
    print(f"Error processing supplier_performance: {e}")

print("Gold Layer Complete! Business analytics tables are successfully built.")

Initializing Gold Layer Analytics...
Aggregating: gold_inventory_summary...
Aggregating: gold_supplier_performance...
Gold Layer Complete! Business analytics tables are successfully built.


In [0]:
from pyspark.sql.functions import col, sum, count, date_trunc

# The same native Unity Catalog path
base_path = "abfss://iaspcontainer@iaspstorage.dfs.core.windows.net"

print("Expanding Gold Layer with Business KPIs...")

try:
    # 1. Load active records from Silver tables
    products = spark.read.format("delta").table("silver_products").filter(col("is_active") == True)
    inventory = spark.read.format("delta").table("silver_inventory").filter(col("is_active") == True)
    transactions = spark.read.format("delta").table("silver_transactions").filter(col("is_active") == True)
    
    # ---------------------------------------------------------
    # KPI 1: Low Stock Alerts
    # ---------------------------------------------------------
    print("Generating Low Stock Alerts...")
    low_stock = inventory.join(products, "product_id") \
        .filter(col("stock_quantity") <= col("reorder_level")) \
        .select("product_id", "product_name", "stock_quantity", "reorder_level", "warehouse_id")
        
    (low_stock.write.format("delta").mode("overwrite")
     .option("path", f"{base_path}/gold/low_stock_alerts/")
     .saveAsTable("gold_low_stock_alerts"))

    # ---------------------------------------------------------
    # KPI 2: Product Movement
    # ---------------------------------------------------------
    print("Generating Product Movement...")
    product_movement = transactions.groupBy("product_id") \
        .agg(
            sum("quantity").alias("total_units_moved"), 
            sum("total_price").alias("total_revenue_generated")
        ) \
        .join(products.select("product_id", "product_name", "category"), "product_id")
        
    (product_movement.write.format("delta").mode("overwrite")
     .option("path", f"{base_path}/gold/product_movement/")
     .saveAsTable("gold_product_movement"))

    # ---------------------------------------------------------
    # KPI 3: Sales Summary (Revenue by Category)
    # ---------------------------------------------------------
    print("Generating Sales Summary...")
    sales_summary = product_movement.groupBy("category") \
        .agg(
            sum("total_revenue_generated").alias("category_revenue"), 
            sum("total_units_moved").alias("category_units_sold")
        )
        
    (sales_summary.write.format("delta").mode("overwrite")
     .option("path", f"{base_path}/gold/sales_summary/")
     .saveAsTable("gold_sales_summary"))

    # ---------------------------------------------------------
    # KPI 4: Sales Trends (Daily Revenue)
    # ---------------------------------------------------------
    print("Generating Sales Trends...")
    sales_trends = transactions.withColumn("sales_date", date_trunc("day", col("transaction_timestamp"))) \
        .groupBy("sales_date") \
        .agg(
            sum("total_price").alias("daily_revenue"), 
            count("transaction_id").alias("transaction_count")
        ) \
        .orderBy("sales_date")
        
    (sales_trends.write.format("delta").mode("overwrite")
     .option("path", f"{base_path}/gold/sales_trends/")
     .saveAsTable("gold_sales_trends"))

    print("Gold Layer KPI Expansion Complete! The data is now fully ready for Power BI.")
    
except Exception as e:
    print(f"Error expanding Gold layer: {e}")

Expanding Gold Layer with Business KPIs...
Generating Low Stock Alerts...
Generating Product Movement...
Generating Sales Summary...
Generating Sales Trends...
Gold Layer KPI Expansion Complete! The data is now fully ready for Power BI.


In [0]:
%sql
SELECT * FROM gold_inventory_summary ORDER BY total_damaged_stock DESC;

product_id,total_stock_available,total_damaged_stock
P1472,51223.0,1057.0
P1101,42829.0,988.0
P1181,33419.0,957.0
P1425,25684.0,881.0
P1219,20955.0,820.0
P1336,42569.0,813.0
P1304,19887.0,810.0
P1125,31712.0,807.0
P1403,43308.0,795.0
P1324,32469.0,793.0


In [0]:
%sql
SELECT * FROM gold_supplier_performance ORDER BY delayed_shipments DESC;

supplier_id,total_shipments,delayed_shipments
S202,271,230
S217,260,229
S201,283,226
S212,261,223
S211,258,221
S219,259,217
S210,257,217
S215,249,216
S200,253,213
S203,248,212
